# Análisis Exploratorio de Datos (EDA) — Superstore

**Objetivo:** Responder las preguntas de negocio definidas en el proyecto:
1. ¿Cómo evolucionaron las ventas y ganancias?
2. ¿Qué categorías/regiones generan más/menos profit?
3. ¿Impacto de los descuentos en las pérdidas?
4. Recomendaciones accionables para mejorar rentabilidad.

## 1. Carga y Validación de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Cargar datos procesados
df = pd.read_csv('../data/processed/superstore_clean.csv', parse_dates=['order_date', 'ship_date'])
print(f"Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Período: {df['order_date'].min().strftime('%Y-%m-%d')} a {df['order_date'].max().strftime('%Y-%m-%d')}")
print(f"\nResumen numérico:")
df[['sales', 'profit', 'quantity', 'discount', 'profit_margin']].describe().round(2)

### Validación rápida

In [ ]:
# Verificar integridad de datos
print("Valores nulos por columna:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "0")
print(f"\nFilas duplicadas: {df.duplicated().sum()}")
print(f"Órdenes únicas: {df['order_id'].nunique()}")
print(f"Clientes únicos: {df['customer_id'].nunique()}")
print(f"Productos únicos: {df['product_id'].nunique()}")

print(f"\nTotales de validación:")
print(f"  Ventas totales:    ${df['sales'].sum():,.2f}")
print(f"  Profit total:      ${df['profit'].sum():,.2f}")
print(f"  Margen promedio:   {df['profit_margin'].mean():.2%}")

---
## 2. Evolución Temporal de Ventas y Ganancias

> **Pregunta de negocio:** ¿Cómo evolucionaron las ventas y ganancias a lo largo del tiempo?

In [ ]:
# Ventas y Profit por año
yearly = df.groupby('order_year')[['sales', 'profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras: Ventas por año
ax1 = axes[0]
bars = ax1.bar(yearly['order_year'], yearly['sales'], color=sns.color_palette('husl', 4))
ax1.set_title('Ventas Totales por Año', fontsize=14, fontweight='bold')
ax1.set_ylabel('Ventas ($)')
ax1.set_xlabel('Año')
for bar, val in zip(bars, yearly['sales']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

# Gráfico de barras: Profit por año
ax2 = axes[1]
colors = ['#2ecc71' if p > 0 else '#e74c3c' for p in yearly['profit']]
bars = ax2.bar(yearly['order_year'], yearly['profit'], color=colors)
ax2.set_title('Profit Total por Año', fontsize=14, fontweight='bold')
ax2.set_ylabel('Profit ($)')
ax2.set_xlabel('Año')
for bar, val in zip(bars, yearly['profit']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Crecimiento interanual
print("Crecimiento interanual:")
for i in range(1, len(yearly)):
    growth_sales = (yearly.iloc[i]['sales'] - yearly.iloc[i-1]['sales']) / yearly.iloc[i-1]['sales'] * 100
    growth_profit = (yearly.iloc[i]['profit'] - yearly.iloc[i-1]['profit']) / yearly.iloc[i-1]['profit'] * 100
    print(f"  {int(yearly.iloc[i-1]['order_year'])} a {int(yearly.iloc[i]['order_year'])}: "
          f"Ventas {growth_sales:+.1f}%, Profit {growth_profit:+.1f}%")

In [ ]:
# Tendencia mensual completa
monthly = df.groupby(['order_year', 'order_month'])[['sales', 'profit']].sum().reset_index()
monthly['period'] = pd.to_datetime(monthly['order_year'].astype(str) + '-' + monthly['order_month'].astype(str) + '-01')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly['period'], monthly['sales'], marker='o', markersize=4, label='Ventas', linewidth=2)
ax.plot(monthly['period'], monthly['profit'], marker='s', markersize=4, label='Profit', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(monthly['period'], monthly['profit'], 0,
                where=monthly['profit'] < 0, alpha=0.3, color='red', label='Pérdidas')
ax.set_title('Evolución Mensual de Ventas y Profit', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Monto ($)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Hallazgos temporales

In [ ]:
# Estacionalidad: ventas promedio por mes
seasonal = df.groupby('order_month')[['sales', 'profit']].mean().reset_index()
months = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, ax = plt.subplots(figsize=(12, 5))
x = range(12)
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], seasonal['sales'], width, label='Ventas promedio', color='#3498db')
bars2 = ax.bar([i + width/2 for i in x], seasonal['profit'], width, label='Profit promedio', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(months)
ax.set_title('Estacionalidad: Ventas y Profit Promedio por Mes', fontsize=14, fontweight='bold')
ax.set_ylabel('Monto promedio ($)')
ax.legend()
plt.tight_layout()
plt.show()

# Identificar mejores y peores meses
best_month = seasonal.loc[seasonal['sales'].idxmax()]
worst_month = seasonal.loc[seasonal['profit'].idxmin()]
print(f"Mejor mes en ventas: {months[int(best_month['order_month'])-1]} (prom. ${best_month['sales']:,.2f})")
print(f"Peor mes en profit: {months[int(worst_month['order_month'])-1]} (prom. ${worst_month['profit']:,.2f})")

### ¿Por qué el profit no siempre acompaña a las ventas?

En el gráfico anterior se observa que **las ventas pueden bajar pero el profit sube** (ej. 2014 → 2015).
Esto se explica por el **product mix effect**: el margen depende de *qué* se vende, no solo de *cuánto*.

In [ ]:
# Margen de ganancia por año
yearly_margin = df.groupby('order_year').agg(
    ventas=('sales', 'sum'),
    profit=('profit', 'sum'),
    descuento_prom=('discount', 'mean')
).round(2)
yearly_margin['margen_%'] = (yearly_margin['profit'] / yearly_margin['ventas'] * 100).round(2)

print("Evolución del margen:")
print(yearly_margin[['ventas', 'profit', 'margen_%', 'descuento_prom']])

# Profit por categoría y año
print("\nProfit por Categoría y Año:")
pivot_cat_year = df.pivot_table(values='profit', index='category', columns='order_year', aggfunc='sum').round(2)
print(pivot_cat_year)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Margen por año
ax1 = axes[0]
bars = ax1.bar(yearly_margin.index, yearly_margin['margen_%'], color=['#3498db', '#2ecc71', '#2ecc71', '#e67e22'])
ax1.set_title('Margen de Ganancia por Año (%)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Margen (%)')
ax1.set_xlabel('Año')
for bar, val in zip(bars, yearly_margin['margen_%']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val}%', ha='center', va='bottom', fontsize=11)

# Profit por categoría apilado por año
ax2 = axes[1]
pivot_cat_year.T.plot(kind='bar', stacked=True, ax=ax2, colormap='Set2')
ax2.set_title('Composición del Profit por Categoría y Año', fontsize=14, fontweight='bold')
ax2.set_ylabel('Profit ($)')
ax2.set_xlabel('Año')
ax2.legend(title='Categoría', fontsize=9)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print("\nConclusión: La diferencia entre ventas y profit se explica por el mix de productos.")
print("Technology mejoró su margen significativamente en 2015, compensando la caída en ventas.")

---
## 3. Rentabilidad por Categoría y Región

> **Pregunta de negocio:** ¿Qué categorías/regiones generan más/menos profit?

In [ ]:
# Profit por Categoría y Sub-Categoría
subcat_profit = df.groupby(['category', 'sub_category'])[['sales', 'profit']].sum().reset_index()
subcat_profit = subcat_profit.sort_values('profit', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if p < 0 else '#2ecc71' for p in subcat_profit['profit']]
bars = ax.barh(subcat_profit['sub_category'], subcat_profit['profit'], color=colors)
ax.set_title('Profit Total por Sub-Categoría', fontsize=14, fontweight='bold')
ax.set_xlabel('Profit ($)')
ax.axvline(x=0, color='black', linewidth=0.8)

# Etiquetas de valor
for bar, val in zip(bars, subcat_profit['profit']):
    x_pos = bar.get_width() + (1000 if val >= 0 else -1000)
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', ha='left' if val >= 0 else 'right', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Resumen
losing = subcat_profit[subcat_profit['profit'] < 0]
print(f"Sub-categorías con pérdidas ({len(losing)}):")
for _, row in losing.iterrows():
    print(f"    {row['sub_category']} ({row['category']}): ${row['profit']:,.2f} "
          f"(sobre ${row['sales']:,.2f} en ventas)")

In [ ]:
# Heatmap: Profit por Región × Categoría
pivot_profit = df.pivot_table(values='profit', index='region', columns='category', aggfunc='sum')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap de Profit
sns.heatmap(pivot_profit, annot=True, fmt=',.0f', cmap='RdYlGn', center=0,
            ax=axes[0], linewidths=1, cbar_kws={'label': 'Profit ($)'})
axes[0].set_title('Profit por Región × Categoría', fontsize=14, fontweight='bold')

# Heatmap de Margen
pivot_margin = df.pivot_table(values='profit_margin', index='region', columns='category', aggfunc='mean')
sns.heatmap(pivot_margin, annot=True, fmt='.1%', cmap='RdYlGn', center=0,
            ax=axes[1], linewidths=1, cbar_kws={'label': 'Margen (%)'})
axes[1].set_title('Margen Promedio por Región × Categoría', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Análisis por Segmento de Cliente
segment = df.groupby('segment')[['sales', 'profit']].agg(['sum', 'mean']).round(2)
segment.columns = ['Ventas Totales', 'Venta Promedio', 'Profit Total', 'Profit Promedio']
segment['Margen'] = (segment['Profit Total'] / segment['Ventas Totales'] * 100).round(2)
segment = segment.sort_values('Profit Total', ascending=False)
print("Rentabilidad por Segmento de Cliente:")
segment

---
## 4. Impacto de los Descuentos en las Pérdidas

> **Pregunta de negocio:** ¿Cuál es el impacto de los descuentos en la rentabilidad?

In [ ]:
# Crear rangos de descuento para análisis
df['discount_range'] = pd.cut(df['discount'],
                               bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
                               labels=['0%', '1-10%', '11-20%', '21-30%', '31-50%', '>50%'])

# Análisis cuantitativo del impacto de descuentos
discount_analysis = df.groupby('discount_range').agg(
    transacciones=('row_id', 'count'),
    ventas_totales=('sales', 'sum'),
    profit_total=('profit', 'sum'),
    profit_promedio=('profit', 'mean'),
    margen_promedio=('profit_margin', 'mean')
).round(2)

discount_analysis['% pérdidas'] = df.groupby('discount_range')['profit'].apply(
    lambda x: (x < 0).mean() * 100
).round(1)

print("Impacto de descuentos en rentabilidad:")
discount_analysis

In [ ]:
# Punto de quiebre: ¿a partir de qué descuento se pierde dinero en promedio?
breakpoint_data = df.groupby('discount')['profit'].mean().reset_index()
breakpoint_data = breakpoint_data.sort_values('discount')

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ecc71' if p >= 0 else '#e74c3c' for p in breakpoint_data['profit']]
ax.bar(breakpoint_data['discount'], breakpoint_data['profit'], color=colors, width=0.03)
ax.axhline(y=0, color='black', linewidth=1)
ax.set_title('Profit Promedio por Nivel de Descuento', fontsize=14, fontweight='bold')
ax.set_xlabel('Descuento')
ax.set_ylabel('Profit Promedio ($)')

# Encontrar el punto de quiebre
negative_discounts = breakpoint_data[breakpoint_data['profit'] < 0]['discount']
if len(negative_discounts) > 0:
    threshold = negative_discounts.min()
    ax.axvline(x=threshold, color='red', linestyle='--', alpha=0.7,
               label=f'Punto de quiebre: {threshold:.0%}')
    ax.legend(fontsize=12)
    print(f"A partir de un descuento del {threshold:.0%}, el profit promedio se vuelve negativo.")

plt.tight_layout()
plt.show()

---
## 5. Top 10 Productos: Más y Menos Rentables

In [ ]:
# Top 10 productos más y menos rentables
product_profit = df.groupby('product_name')[['sales', 'profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 10 más rentables
top10 = product_profit.nlargest(10, 'profit')
axes[0].barh(top10['product_name'], top10['profit'], color='#2ecc71')
axes[0].set_title('Top 10 Productos Más Rentables', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Profit ($)')
for i, (_, row) in enumerate(top10.iterrows()):
    axes[0].text(row['profit'] + 100, i, f"${row['profit']:,.0f}", va='center', fontsize=9)

# Top 10 menos rentables
bottom10 = product_profit.nsmallest(10, 'profit')
axes[1].barh(bottom10['product_name'], bottom10['profit'], color='#e74c3c')
axes[1].set_title('Top 10 Productos Menos Rentables', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Profit ($)')
for i, (_, row) in enumerate(bottom10.iterrows()):
    axes[1].text(row['profit'] - 100, i, f"${row['profit']:,.0f}", va='center', ha='right', fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Conclusiones y Recomendaciones

### Hallazgos Clave

**1. Tendencia general:** Las ventas crecieron de $484K (2014) a $733K (2017), un crecimiento acumulado del 51.4%. El profit también creció de $49K a $93K (+88.6%). Sin embargo, el crecimiento no fue lineal ni proporcional:
- En 2015, las ventas bajaron un 2.8% pero el profit subió un 24.4%. Technology mejoró su rentabilidad de $21K a $33K (+55.9%) vendiendo productos de mayor margen.
- El margen de ganancia mejoró de 10.23% (2014) a 13.43% (2016), pero retrocedió a 12.74% en 2017, indicando que el crecimiento en ventas se logró parcialmente a costa del margen.
- Se observa estacionalidad: noviembre y diciembre concentran los picos de ventas y profit en los 4 años.

**2. Categorías problemáticas:** De las 17 sub-categorías, 3 generan pérdidas netas:
- **Tables** (Furniture): la sub-categoría más deficitaria, con pérdidas significativas a pesar de un alto volumen de ventas. Esto indica un problema de pricing o descuentos, no de demanda.
- **Bookcases** (Furniture): pérdidas menores pero consistentes.
- **Supplies** (Office Supplies): margen negativo marginal.
- **Technology** es la categoría más rentable, liderada por Copiers y Phones. Furniture es la más problemática: solo Chairs y Furnishings son rentables.
- La región **Central** muestra los márgenes más bajos, particularmente en Furniture. Las regiones West y East son las más rentables.

**3. Descuentos:** Existe una correlación negativa clara entre descuento y profit:
- Sin descuento (0%): el profit promedio por transacción es positivo y estable.
- Descuentos de 1-20%: el profit se reduce pero sigue siendo positivo.
- A partir del 30%: el profit promedio se vuelve negativo (punto de quiebre).
- Descuentos del 50% o más: prácticamente todas las transacciones generan pérdidas.

**4. Regiones:** La región Central es la menos rentable del negocio, especialmente en Furniture donde los márgenes son negativos. Las regiones West y East lideran en rentabilidad.

**5. Productos:** Los 10 productos más rentables pertenecen mayoritariamente a Technology (copiadoras, teléfonos). Los 10 menos rentables incluyen mesas y máquinas específicas con pérdidas acumuladas significativas. Estos productos deberían revisarse individualmente para determinar si es viable continuar vendiéndolos.

### Recomendaciones Accionables

1. **Establecer un tope máximo de descuento del 20-25%** como política comercial. Los descuentos superiores destruyen valor consistentemente.
2. **Revisar la estrategia de pricing de Tables y Bookcases** (Furniture). Se vende mucho pero se pierde dinero, lo que sugiere descuentos excesivos o costos altos.
3. **Investigar la región Central**, que presenta los márgenes más bajos. Determinar si es por mix de productos, descuentos locales, o costos operativos u otros factores.
4. **Monitorear el margen en 2017**: el crecimiento en ventas no se tradujo proporcionalmente en profit, indicando presión sobre los márgenes que podría agravarse.